In [3]:
from pathlib import Path
import pandas as pd
import pyreadstat

In [4]:
ROOT = Path(r"C:\Users\Carl\Python\Projects\econometrics\data\UgandaLSMS\Wave_8")
OUTPUT_FILE = ROOT / "wave8_variable_inventory.xlsx"


def infer_survey_type(file_stem: str) -> str:
    if file_stem.startswith("GSEC"):
        return "household"
    elif file_stem.startswith("AGSEC"):
        return "agriculture"
    elif file_stem.startswith("CSEC"):
        return "community"
    return "unknown"


def infer_section(file_stem: str) -> str:
    return file_stem


def detect_id_type(var_name: str, var_label: str) -> tuple[str, dict]:
    name = (var_name or "").strip().lower()
    label = (var_label or "").strip().lower()
    text = f"{name} {label}"

    is_hhid_var = (
        name == "hhid"
        or "unique hh identifier" in label
        or "household identifier" in label
        or "household id" in label
    )

    is_pid_var = (
        name == "pid"
        or "unique person identifier" in label
        or "person identifier" in label
        or "person id" in label
    )

    is_plot_id_var = (
        name in {"plotid", "pltid"}
        or "plot id" in text
        or "plot identifier" in text
    )

    is_parcel_id_var = (
        name in {"parcelid", "prcid", "a3q1"}
        or "parcel id" in text
        or "parcel identifier" in text
    )

    is_village_id_var = (
        name in {"villageid", "ea", "commid"}
        or "village id" in text
        or "community id" in text
        or "enumeration area" in text
    )

    key_type_guess = None
    if is_hhid_var:
        key_type_guess = "household_id"
    elif is_pid_var:
        key_type_guess = "person_id"
    elif is_plot_id_var:
        key_type_guess = "plot_id"
    elif is_parcel_id_var:
        key_type_guess = "parcel_id"
    elif is_village_id_var:
        key_type_guess = "village_id"

    flags = {
        "is_hhid_var": is_hhid_var,
        "is_pid_var": is_pid_var,
        "is_plot_id_var": is_plot_id_var,
        "is_parcel_id_var": is_parcel_id_var,
        "is_village_id_var": is_village_id_var,
    }

    return key_type_guess, flags


def extract_dta_metadata(file_path: Path, wave_name: str = "Wave_8") -> pd.DataFrame:
    df, meta = pyreadstat.read_dta(file_path)

    column_names = list(df.columns)
    column_labels = meta.column_labels if meta.column_labels else [""] * len(column_names)

    rows = []
    for i, var_name in enumerate(column_names):
        var_label = column_labels[i] if i < len(column_labels) else ""
        key_type_guess, flags = detect_id_type(var_name, var_label)

        rows.append({
            "wave": wave_name,
            "survey_type": infer_survey_type(file_path.stem),
            "section": infer_section(file_path.stem),
            "file_name": file_path.name,
            "file_stem": file_path.stem,
            "var_name": var_name,
            "var_label": var_label,
            "order_in_file": i + 1,
            "key_type_guess": key_type_guess,
            **flags,
            "notes": ""
        })

    meta_df = pd.DataFrame(rows)

    # section-level inherited identifiers
    meta_df["has_hhid"] = meta_df["is_hhid_var"].any()
    meta_df["has_pid"] = meta_df["is_pid_var"].any()
    meta_df["has_plot_id"] = meta_df["is_plot_id_var"].any()
    meta_df["has_parcel_id"] = meta_df["is_parcel_id_var"].any()
    meta_df["has_village_id"] = meta_df["is_village_id_var"].any()

    return meta_df


def infer_entity_level(row):
    if row["has_parcel_id"]:
        return "parcel"
    if row["has_plot_id"]:
        return "plot"
    if row["has_pid"]:
        return "person"
    if row["has_hhid"]:
        return "household"
    if row["has_village_id"]:
        return "community"
    return "unknown"


def main():
    dta_files = sorted(ROOT.glob("*.dta"))
    all_meta = []

    for file_path in dta_files:
        try:
            meta_df = extract_dta_metadata(file_path)
            all_meta.append(meta_df)
            print(f"Done: {file_path.name}")
        except Exception as e:
            print(f"Failed: {file_path.name} -> {e}")

    if not all_meta:
        print("No metadata extracted.")
        return

    final_df = pd.concat(all_meta, ignore_index=True)
    final_df["entity_level"] = final_df.apply(infer_entity_level, axis=1)

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        final_df.to_excel(writer, sheet_name="variables", index=False)

        summary = (
            final_df.groupby(["survey_type", "section"])
            .agg(
                n_variables=("var_name", "count"),
                has_hhid=("has_hhid", "max"),
                has_pid=("has_pid", "max"),
                has_plot_id=("has_plot_id", "max"),
                has_parcel_id=("has_parcel_id", "max"),
                has_village_id=("has_village_id", "max"),
                entity_level=("entity_level", "first")
            )
            .reset_index()
            .sort_values(["survey_type", "section"])
        )
        summary.to_excel(writer, sheet_name="summary", index=False)

    print(f"\nSaved inventory to:\n{OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Done: AGSEC1.dta
Done: AGSEC10.dta
Done: AGSEC11.dta
Done: AGSEC2A.dta
Done: AGSEC2B.dta
Done: AGSEC3A.dta
Done: AGSEC3A_1.dta
Done: AGSEC3B.dta
Done: AGSEC3B_1.dta
Done: AGSEC4A.dta
Done: AGSEC4B.dta
Done: AGSEC5A.dta
Done: AGSEC5B.dta
Done: AGSEC6A.dta
Done: AGSEC6B.dta
Done: AGSEC6C.dta
Done: AGSEC7.dta
Done: AGSEC8A.dta
Done: AGSEC8B.dta
Done: AGSEC8C.dta
Done: AGSEC9A.dta
Done: AGSEC9B.dta
Done: CSEC11_0.dta
Done: CSEC1A.dta
Done: CSEC2.dta
Done: CSEC2A.dta
Done: CSEC2B.dta
Done: CSEC2C.dta
Done: CSEC2C_0.dta
Done: CSEC3_0.dta
Done: CSEC3A.dta
Done: CSEC3B.dta
Done: CSEC3C.dta
Done: CSEC3D.dta
Done: CSEC3E.dta
Done: CSEC3F.dta
Done: CSEC3G.dta
Done: CSEC3H.dta
Done: CSEC3I.dta
Done: CSEC3J.dta
Done: CSEC3K.dta
Done: CSEC3L.dta
Done: CSEC3M.dta
Done: CSEC4A.dta
Done: CSEC4B.dta
Done: CSEC4C.dta
Done: CSEC4D.dta
Done: CSEC4E.dta
Done: CSEC4F.dta
Done: CSEC4G.dta
Done: CSEC4H_1.dta
Done: CSEC4I.dta
Done: CSEC4J.dta
Done: CSEC4K.dta
Done: CSEC4L.dta
Done: CSEC4M.dta
Done: CSEC4N.dta
D